# **Genetic Algorithm - One-max Problem**
### Quang-Huy Tran, Student at HCMC University of Technology - VNU-HCM.

- One-max là một bài toán tối ưu trong đó mỗi cá thể được biểu diễn bằng một
chuỗi nhị phân (gồm các số 0 và 1). Mục tiêu là tìm ra chuỗi có nhiều số 1 nhất.  
- Trải qua các quá trình: Chọn lọc (Selection + Elitism) - Lai ghép (Crossover) - Đột biến (Mutation).

In [1]:
import numpy as np

In [ ]:
# Initial population.
# each row is individual, each individual contains N gene.
np.random.seed(42)
population = np.random.randint(0, 2, size=(5, 5))
population

array([[0, 1, 0, 0, 0],
       [1, 0, 0, 0, 1],
       [0, 0, 0, 0, 1],
       [0, 1, 1, 1, 0],
       [1, 0, 1, 1, 1]])

In [ ]:
# fitness function -- using 'sum' for one-max problem
def fitness(pop):
  return np.sum(pop, axis=1)

fitness(population)

array([1, 2, 1, 3, 4])

In [ ]:
# selection step example
score = fitness(population)
elitism = 2
max_idx_01, max_idx_02 = np.argsort(score)[-elitism:]
max_ind_01, max_ind_02 = population[max_idx_02], population[max_idx_01]
np.vstack((max_ind_01, max_ind_02))

array([[1, 0, 1, 1, 1],
       [0, 1, 1, 1, 0]])

In [ ]:
# reuse elitism
score = fitness(population)
elitism = 2
max_idx_01, max_idx_02 = np.argsort(score)[-elitism:]
max_ind_01, max_ind_02 = population[max_idx_02], population[max_idx_01]
selection_result = np.vstack((max_ind_01, max_ind_02))

# Choosing two individuals randomly
num_select = 2
for i in range(num_select):
  a, b = np.random.randint(0, 5, size=(2,))
  ind_01 = population[a]
  ind_02 = population[b]
  selection_result = np.vstack((selection_result, ind_01)) if score[a] > score[b] else np.vstack((selection_result, ind_02))

selection_result

array([[1, 0, 1, 1, 1],
       [0, 1, 1, 1, 0],
       [0, 1, 1, 1, 0],
       [1, 0, 1, 1, 1]])

In [6]:
# encapsulation
def selection(population, num_select=2, elitism=2):
  max_length = len(population)
  score = fitness(population)
  elite_idx = np.argsort(score)[-elitism:]
  results = [population[i] for i in elite_idx]

  for i in range(num_select):
    a, b = np.random.randint(0, max_length, size=(2,))
    ind_01 = population[a]
    ind_02 = population[b]
    selection_result = results.append(ind_01) if score[a] > score[b] else results.append(ind_02)

  return np.array(results)

selection(population)

array([[0, 1, 1, 1, 0],
       [1, 0, 1, 1, 1],
       [0, 1, 1, 1, 0],
       [1, 0, 1, 1, 1]])

In [ ]:
# crossover step
selection_results = selection(population)
len_selection = len(selection_results)
len_gene = len(selection_results[0])
for i in range(len_selection):
  idx_01, idx_02 = np.random.randint(0, len_selection, size=(2,))
  ind_01, ind_02 = selection_results[idx_01], selection_results[idx_02]
  crossover_rate = np.random.random((len_gene,))
  print(f"rate: {crossover_rate}")

  # logging
  print("Before:")
  print(f"Individual 1: {ind_01}")
  print(f"Individual 2: {ind_02}")

  for j in range(len_gene):
    if crossover_rate[j] < 0.9:
      ind_01[j], ind_02[j] = ind_02[j], ind_01[j]

  # logging
  print("After:")
  print(f"Individual 1: {ind_01}")
  print(f"Individual 2: {ind_02}")
  print(f"---------------")



rate: [0.04666566 0.97375552 0.23277134 0.09060643 0.61838601]
Before:
Individual 1: [0 1 0 0 0]
Individual 2: [1 0 1 1 1]
After:
Individual 1: [1 1 1 1 1]
Individual 2: [0 0 0 0 0]
---------------
rate: [0.98323089 0.46676289 0.85994041 0.68030754 0.45049925]
Before:
Individual 1: [0 0 0 0 1]
Individual 2: [1 1 1 1 1]
After:
Individual 1: [0 1 1 1 1]
Individual 2: [1 0 0 0 1]
---------------
rate: [0.94220176 0.56328822 0.3854165  0.01596625 0.23089383]
Before:
Individual 1: [0 0 0 0 0]
Individual 2: [0 1 1 1 1]
After:
Individual 1: [0 1 1 1 1]
Individual 2: [0 0 0 0 0]
---------------
rate: [0.68326352 0.60999666 0.83319491 0.17336465 0.39106061]
Before:
Individual 1: [0 0 0 0 0]
Individual 2: [1 0 0 0 1]
After:
Individual 1: [1 0 0 0 1]
Individual 2: [0 0 0 0 0]
---------------


In [19]:
# encapsulation
def crossover(population, pool, rate=0.9):
  max_length = len(population)
  results = []
  len_selection = len(pool)
  len_gene = len(pool[0])

  for i in range(len_selection):
    idx_01, idx_02 = np.random.randint(0, len_selection, size=(2,))
    ind_01, ind_02 = pool[idx_01], pool[idx_02]
    crossover_rate = np.random.random((len_gene,))

    for j in range(len_gene):
      if crossover_rate[j] < rate:
        ind_01[j], ind_02[j] = ind_02[j], ind_01[j]

    if max_length - len(results) == 2:
      results.append(ind_01)
      results.append(ind_02)
      break

    if max_length - len(results) == 1:
      random_select = np.random.randint(0, 2)
      results.append(ind_01) if random_select == 0 else results.append(ind_02)
      break

    results.append(ind_01)
    results.append(ind_02)

  return np.array(results)

pool = selection(population)
print(f'pool before:\n {pool}')
crossover(population, pool)
print(f'pool after:\n {pool}')
print("Quan sát, pool đã bị thay đổi.")

pool before:
 [[0 1 1 1 0]
 [1 0 1 1 1]
 [1 0 1 1 1]
 [0 1 1 1 0]]
pool after:
 [[1 0 1 1 1]
 [0 1 1 1 0]
 [0 1 1 1 0]
 [1 0 1 1 1]]
Quan sát, pool đã bị thay đổi.


### với hàm `crossover` ở trên, ta thấy rằng: pool đã thay đổi khi đi qua `crossover`.  
Đây là hành vi không mong muốn, vì khi thực hiện crossover ta cần chọn 2 cá thể nguyên vẹn từ pool, chứ không phải dữ liệu thay đổi.  
Đây là lí do cần `copy`.

In [20]:
import copy

# đóng gói
def crossover(population, pool, rate=0.9):
  max_length = len(population)
  results = []
  len_selection = len(pool)
  len_gene = len(pool[0])

  for i in range(len_selection):
    idx_01, idx_02 = np.random.randint(0, len_selection, size=(2,))
    ind_01, ind_02 = copy.deepcopy(pool[idx_01]), copy.deepcopy(pool[idx_02])
    crossover_rate = np.random.random((len_gene,))

    for j in range(len_gene):
      if crossover_rate[j] < rate:
        ind_01[j], ind_02[j] = ind_02[j], ind_01[j]

    if max_length - len(results) == 2:
      results.append(ind_01)
      results.append(ind_02)
      break

    if max_length - len(results) == 1:
      random_select = np.random.randint(0, 2)
      results.append(ind_01) if random_select == 0 else results.append(ind_02)
      break

    results.append(ind_01)
    results.append(ind_02)

  return np.array(results)

pool = selection(population)
print(f'pool before:\n {pool}')
crossover(population, pool)
print(f'pool after:\n {pool}')
print("Pool đã ổn định.")

pool before:
 [[0 1 1 1 0]
 [1 0 1 1 1]
 [0 1 1 1 0]
 [0 1 1 1 0]]
pool after:
 [[0 1 1 1 0]
 [1 0 1 1 1]
 [0 1 1 1 0]
 [0 1 1 1 0]]
Pool đã ổn định.


In [ ]:
# mutate w rate ~0.05
mutation_threshold = 0.95
new_pop = copy.deepcopy(crossover(population, pool))
len_pop = len(new_pop)
len_gene = len(new_pop[0])
for i in range(len_pop):
  mutation_rate = np.random.random((len_gene,))
  print(f"Individual before mutation: {new_pop[i]}")
  for j in range(len_gene):
    if mutation_rate[j] > mutation_threshold:
      new_pop[i][j] = 1 - new_pop[i][j]

  print(f"Individual after mutation: {new_pop[i]}")



Individual before mutation: [0 1 1 1 0]
Individual after mutation: [0 1 1 1 0]
Individual before mutation: [0 1 1 1 0]
Individual after mutation: [0 1 1 1 0]
Individual before mutation: [1 0 1 1 1]
Individual after mutation: [1 0 1 1 1]
Individual before mutation: [0 1 1 1 0]
Individual after mutation: [0 1 1 1 0]
Individual before mutation: [0 1 1 1 0]
Individual after mutation: [0 1 1 0 0]


In [33]:
# encapsulation
def mutation(new_generation, threshold=0.95):
  new_pop = copy.deepcopy(new_generation)
  len_pop = len(new_pop)
  len_gene = len(new_pop[0])
  for i in range(len_pop):
    mutation_rate = np.random.random((len_gene,))
    # print(f"Individual before mutation: {new_pop[i]}")
    for j in range(len_gene):
      if mutation_rate[j] > threshold:
        new_pop[i][j] = 1 - new_pop[i][j]

    # print(f"Individual after mutation: {new_pop[i]}")
  return new_pop

next_generation = crossover(population, pool)
mutation(next_generation)

array([[0, 1, 1, 1, 0],
       [0, 1, 0, 1, 0],
       [1, 1, 1, 1, 1],
       [0, 0, 1, 1, 0],
       [0, 1, 1, 1, 0]])

In [ ]:
# all step
random_population = np.random.randint(0, 2, size=(5, 5))
print(f'Begin: \n{random_population}')
print(f'Begin Score: {fitness(random_population)}')
print('--------------------')
selection_result = selection(random_population)
print(f'After selection step: \n{selection_result}')
print('--------------------')
crossover_result = crossover(random_population, selection_result)
print(f'After crossover step: \n{crossover_result}')
print('--------------------')
mutation_result = mutation(crossover_result)
print(f'After mutation step: \n{mutation_result}')
print('--------------------')
print(f'Next Score: {fitness(mutation_result)}')


Begin: 
[[1 1 1 1 0]
 [1 1 0 1 0]
 [1 1 0 0 1]
 [1 1 0 1 0]
 [0 1 0 1 1]]
Begin Score: [4 3 3 3 3]
--------------------
After selection step: 
[[0 1 0 1 1]
 [1 1 1 1 0]
 [1 1 0 0 1]
 [1 1 1 1 0]]
--------------------
After crossover step: 
[[1 1 1 1 0]
 [0 1 0 1 1]
 [1 1 1 1 0]
 [0 1 0 1 1]
 [1 1 1 1 0]]
--------------------
After mutation step: 
[[1 1 0 1 0]
 [0 1 0 0 1]
 [1 1 1 1 0]
 [0 1 0 1 1]
 [1 1 1 1 0]]
--------------------
Next Score: [3 2 4 3 4]


In [ ]:
# encapsulation
def fitness(pop):
  return np.sum(pop, axis=1)


def selection(population, num_select=2, elitism=2):
  max_length = len(population)
  score = fitness(population)
  elite_idx = np.argsort(score)[-elitism:]
  results = [population[i] for i in elite_idx]

  for i in range(num_select):
    a, b = np.random.randint(0, max_length, size=(2,))
    ind_01 = population[a]
    ind_02 = population[b]
    selection_result = results.append(ind_01) if score[a] > score[b] else results.append(ind_02)

  return np.array(results)


def crossover(population, pool, rate=0.9):
  max_length = len(population)
  results = []
  len_selection = len(pool)
  len_gene = len(pool[0])

  for i in range(len_selection):
    idx_01, idx_02 = np.random.randint(0, len_selection, size=(2,))
    ind_01, ind_02 = copy.deepcopy(pool[idx_01]), copy.deepcopy(pool[idx_02])
    crossover_rate = np.random.random((len_gene,))

    for j in range(len_gene):
      if crossover_rate[j] < rate:
        ind_01[j], ind_02[j] = ind_02[j], ind_01[j]

    if max_length - len(results) == 2:
      results.append(ind_01)
      results.append(ind_02)
      break

    if max_length - len(results) == 1:
      random_select = np.random.randint(0, 2)
      results.append(ind_01) if random_select == 0 else results.append(ind_02)
      break

    results.append(ind_01)
    results.append(ind_02)

  return np.array(results)


def mutation(new_generation, threshold=0.95):
  new_pop = copy.deepcopy(new_generation)
  len_pop = len(new_pop)
  len_gene = len(new_pop[0])
  for i in range(len_pop):
    mutation_rate = np.random.random((len_gene,))
    for j in range(len_gene):
      if mutation_rate[j] > threshold:
        new_pop[i][j] = 1 - new_pop[i][j]

  return new_pop


def genetic_implement(population, epoch=10):
  for i in range(epoch):
    selection_result = selection(population)
    crossover_result = crossover(population, selection_result)
    population = mutation(crossover_result)
    print(f"POPULATION AFTER EPOCH {i+1}: \n{population}")
    print(f"SCORE AFTER EPOCH {i+1}: {fitness(population)}\n")
    print("------------------------------")

  return population


In [60]:
population = np.zeros((5, 5), dtype=np.int32)
print(f'Begin: \n{population}')
print(f'Begin Score: {fitness(population)}')
print('--------------------')
genetic_implement(population, epoch=20)

Begin: 
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Begin Score: [0 0 0 0 0]
--------------------
POPULATION AFTER EPOCH 1: 
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
SCORE AFTER EPOCH 1: [0 0 0 0 0]

------------------------------
POPULATION AFTER EPOCH 2: 
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
SCORE AFTER EPOCH 2: [0 0 0 0 0]

------------------------------
POPULATION AFTER EPOCH 3: 
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
SCORE AFTER EPOCH 3: [0 0 0 0 0]

------------------------------
POPULATION AFTER EPOCH 4: 
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
SCORE AFTER EPOCH 4: [0 0 0 0 0]

------------------------------
POPULATION AFTER EPOCH 5: 
[[0 0 0 0 1]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]]
SCORE AFTER EPOCH 5: [1 0 0 1 0]

------------------------------
POPULATION AFTER EPOCH 6: 
[[0 1 0 0 0]
 [0 1 0 0 0]
 [0 0 0 1 1]
 [0 0 0 0 1]
 [0 0 0 0 1

array([[1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1]], dtype=int32)